# 🍽️ Gemini AI - Restaurant Agent Tools Setup

---

## 📚 Notebook Overview

This notebook demonstrates how to build an **AI-powered multi-tool conversational agent** for a **restaurant use case** using **Google’s Gemini API** with **function calling** support. It integrates **custom tool execution logic** and **response evaluation** using the `llumo` Python library.

---

## ✅ What You'll Find in This Notebook

### 🔧 Tool Definitions
- `getMenu()` – Returns the available menu with item details
- `addToCart(item, quantity)` – Adds items to the cart if stock is available
- `removeFromCart(item, quantity)` – Removes items from the cart and updates stock
- `getOrderDetails()` – Places an order, generates a unique order ID, and clears the cart
- `clearCart()` – Clears the cart and restores item stock
- `viewOrderHistory()` – Displays the user’s past orders

These tools simulate realistic restaurant operations.

---

### 🤖 Agent Interaction & Tool Calling
- Uses **Google Gemini API** with function calling capability
- Parses user queries and automatically selects the appropriate tool
- Tool outputs are appended to the chat history for better context tracking

---

### 🔁 Multi-Turn Conversation Loop
- Accepts multiple user queries in sequence
- Agent responds, tool gets triggered if needed, response is recorded
- Chat history is maintained and updated throughout the session

---

### 📊 Data Logging
- All interactions (queries, tool calls, responses, chat history) are stored in a **Pandas DataFrame**
- Useful for tracking behavior and reviewing conversation flows

---

### 🧪 Response Evaluation with `llumo`
- Uses the `llumo` Python library to **evaluate LLM-generated responses**

---

## 🧠 Why Use This Notebook?
This notebook is ideal for:
- Building tool-augmented AI agents
- Practicing function calling with Gemini
- Collecting structured conversation data
- Evaluating agent performance using standardized metrics

---


# ⚙️ **Install Dependencies**




In [10]:
!pip install llumo -q
!pip install google -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 3.6 MB/s eta 0:00:00


# **⬇ Import necessary modules**


In [ ]:
import json         # For parsing and handling JSON data
import uuid         # For generating unique order IDs
from google import genai                      # Import the main genai module
from google.genai import types               # Import the types needed for tool and message handling


#**🔑 Setting API Keys as Environment Variables**





In [1]:
import os

# Set your OpenAI API Key
os.environ["GOOGLE_API_KEY"] = "Enter Your Google API Key"

# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = "Enter Your LLumo API Key"

google_key = os.getenv("GOOGLE_API_KEY")
llumo_key = os.getenv("LLUMO_API_KEY")

# **🍽️ Define Function Tools for the Agent**
- 🍔 This block defines core restaurant operations like viewing the menu, managing the cart, and handling order history using Python functions.  
- 🛒 It simulates a mini ordering system with item stock updates, order placement using unique IDs, and retrieval of past orders.


In [11]:
import uuid  # Import uuid module for generating unique order IDs

# Menu dictionary with items, their price, stock quantity, and description
menu = {
    "burger": {"price": 150, "stock": 10, "description": "Delicious beef burger"},
    "pizza": {"price": 300, "stock": 5, "description": "Cheesy pepperoni pizza"},
    "pasta": {"price": 250, "stock": 8, "description": "Creamy alfredo pasta"},
    "coke": {"price": 50, "stock": 20, "description": "Refreshing soft drink"},
    "sandwich": {"price": 120, "stock": 15, "description": "Grilled cheese sandwich"},
    "fries": {"price": 100, "stock": 12, "description": "Crispy golden french fries"},
    "mojito": {"price": 180, "stock": 10, "description": "Cool mint mojito"},
    "coffee": {"price": 120, "stock": 20, "description": "Hot brewed coffee"},
    "tea": {"price": 80, "stock": 25, "description": "Refreshing herbal tea"}
}

# Cart to hold items user wants to buy
cart = {}

# Order history to store past orders with order IDs
orderHistory = {}

def getMenu():
    # Return the current menu as a string
    return str(menu)

def addToCart(item, quantity):
    # Add given quantity of item to cart if available in stock
    item = item.lower()
    if item in menu:
        if menu[item]["stock"] >= quantity:
            cart[item] = cart.get(item, 0) + quantity  # Increase quantity in cart
            menu[item]["stock"] -= quantity  # Reduce stock accordingly
            return str({"message": f"{quantity} {item}(s) added to cart.", "cart": cart})
        else:
            # Not enough stock available
            return str({"error": f"Only {menu[item]['stock']} {item}(s) available."})
    # Item not found in menu
    return str({"error": "Item not available in menu."})

def removeFromCart(item, quantity):
    # Remove given quantity of item from cart, update stock accordingly
    item = item.lower()
    if item in cart:
        if cart[item] > quantity:
            cart[item] -= quantity  # Decrease quantity in cart
            menu[item]["stock"] += quantity  # Increase stock accordingly
            return str({"message": f"{quantity} {item}(s) removed from cart.", "cart": cart})
        else:
            # Remove item completely if quantity to remove >= in cart
            menu[item]["stock"] += cart[item]  # Restore full quantity to stock
            del cart[item]
            return str({"message": f"{item} removed from cart.", "cart": cart})
    # Item not found in cart
    return str({"error": "Item not in cart."})

def getOrderDetails():
    # Generate order details and order ID, clear cart after placing order
    if not cart:
        return str({"message": "Your cart is empty."})
    # Calculate total cost based on price and quantity
    total = sum(menu[item]["price"] * qty for item, qty in cart.items())
    # Generate a short unique order ID
    order_id = str(uuid.uuid4())[:8]
    # Store order details in orderHistory
    orderHistory[order_id] = {"cart": cart.copy(), "total": total}
    cart.clear()  # Clear cart after order placed
    return str({"orderId": order_id, "order": orderHistory[order_id]})

def clearCart():
    # Clear cart and restore stock for all items in cart
    for item, qty in cart.items():
        menu[item]["stock"] += qty
    cart.clear()
    return str({"message": "Cart has been cleared."})

def viewOrderHistory():
    # Return past orders or message if no past orders found
    return str(orderHistory) if orderHistory else str({"message": "No past orders."})


#**🧠🔧 Gemini AI Tool Setup for Restaurant Ordering Agent**
These tools follow Google's function declaration format


In [12]:
# 📦 Import necessary modules
from google import genai                          # Main Google GenAI SDK
from google.genai import types                   # For defining tools and content structures
import os                                         # To access environment variables


# 📋 Tool: Get Menu
menuTool = {
    "name": "getMenu",
    "description": "Get the restaurant menu.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

# ➕ Tool: Add Item to Cart
cartAddTool = {
    "name": "addToCart",
    "description": "Add an item to the cart.",
    "parameters": {
        "type": "object",
        "properties": {
            "item": {"type": "string"},              # Name of the item to add
            "quantity": {"type": "integer"}          # Quantity of the item
        },
        "required": ["item", "quantity"]
    }
}

# ➖ Tool: Remove Item from Cart
cartRemoveTool = {
    "name": "removeFromCart",
    "description": "Remove an item from the cart.",
    "parameters": {
        "type": "object",
        "properties": {
            "item": {"type": "string"},              # Name of the item to remove
            "quantity": {"type": "integer"}          # Quantity to remove
        },
        "required": ["item", "quantity"]
    }
}

# 🧾 Tool: Get Order Details
orderDetailsTool = {
    "name": "getOrderDetails",
    "description": "Get the order details and generate an order ID.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

# 🧹 Tool: Clear Cart
clearCartTool = {
    "name": "clearCart",
    "description": "Clear all items from the cart.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

# 📜 Tool: View Order History
orderHistoryTool = {
    "name": "viewOrderHistory",
    "description": "View past order history.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

# 🧰 Register All Tools with Gemini
tools = types.Tool(function_declarations=[
    menuTool,
    cartAddTool,
    cartRemoveTool,
    orderDetailsTool,
    clearCartTool,
    orderHistoryTool
])


# **🗂️🔍 Tool Descriptions Dictionary**


In [4]:

# Dictionary mapping each tool's name to its purpose/description.
tool_descriptions = {
    "getMenu": "Get the restaurant menu.",
    "addToCart": "Add an item to the cart.",
    "removeFromCart": "Remove an item from the cart.",
    "getOrderDetails": "Get the order details and generate an order ID.",
    "clearCart": "Clear all items from the cart.",
    "viewOrderHistory": "View past order history."
}


# **🛠️📞 Handling Tool Call Execution - for Gemini Responses**
🔧 This function handles the execution of Gemini-generated tool calls by invoking the correct restaurant tool function.  
🧠 It updates the chat history with both the function call and the corresponding tool response using `google.genai.types`.


In [13]:

import json
from google.genai import types

# 📦 Executes tool calls generated by the Gemini model and updates the chat history
def executeToolCall(toolCall, chat_history):
    for tool in toolCall:
        # 🧠 Add the model's function call (tool invocation) to the history
        chat_history.append(types.Content(role="model", parts=[types.Part(function_call=tool.function_call)]))

        # 🏷️ Extract tool name and arguments from the function call
        tool_name = tool.function_call.name
        args = tool.function_call.args

        # 🧾 Parse stringified JSON arguments if needed
        if isinstance(args, str):
            args = json.loads(args)

        # 🚀 Execute the corresponding tool function based on the tool name
        if tool_name == "getMenu":
            func_response = getMenu()

        elif tool_name == "addToCart":
            func_response = addToCart(args["item"], args["quantity"])

        elif tool_name == "removeFromCart":
            func_response = removeFromCart(args["item"], args["quantity"])

        elif tool_name == "getOrderDetails":
            func_response = getOrderDetails()

        elif tool_name == "clearCart":
            func_response = clearCart()

        elif tool_name == "viewOrderHistory":
            func_response = viewOrderHistory()

        else:
            # ❌ If the tool is not recognized, return a fallback response
            func_response = f"Tool `{tool_name}` not implemented."

        # ✅ Add the tool's function response to the chat history
        tool_response = types.Part.from_function_response(
            name=tool_name,
            response={"result": str(func_response)}  # Ensure response is stringified
        )
        chat_history.append(types.Content(role="tool", parts=[tool_response]))

    return chat_history


# **💬🤖 Simulating Multi-Turn Agent Conversation in Gemini**


📌 Note:
The input data for agent evaluation must have following mandatory keys for each query result:

- query

- output

- messageHistory

- tools


```
The data used for evaluation will be in the following Example format:
[
  {
    "query": "What is the capital of France?",
    "output": "The capital of France is Paris.",
    "messageHistory": '''[{"role": "user", "content": "What is the capital of France?"}, {"role": "assistant", "content": "The capital of France is Paris."}]''',
    "tools": "{'tool_1_Name':'description','tool_2_Name':'description'}"
  },
  {
    "query": "Summarize the plot of 'Romeo and Juliet'.",
    "output": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families.",
    "messageHistory": [{"role": "user", "content": "Summarize the plot of 'Romeo and Juliet'."}, {"role": "assistant", "content": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families."}],
    "tools": "{'tool_1_Name':'description','tool_2_Name':'description'}"
  }
]
```

In [6]:
#  📜 List to store results of each user interaction
results = []

In [7]:

import time

# 🧠 User queries to simulate interactions with the agent
user_queries = [
    "Show me the menu ",
    "Add 2 burgers to my cart",
    "Show me the order history",  # 🔧 Missing comma was added here
]

# 🔁 Loop through each user query and simulate conversation
for query in user_queries:
    time.sleep(4)  # ⏱️ Optional delay between requests

    # 🔐 Initialize Gemini client with API key
    client = genai.Client(api_key=google_key)

    # 📨 Prepare initial user message content
    contents = [
        types.Content(role="user", parts=[types.Part(text=query)])
    ]

    # ⚙️ Prepare config with available tools
    config = types.GenerateContentConfig(tools=[tools])

    # 🤖 First model call to interpret the user query and suggest tool calls
    response = client.models.generate_content(
        model="gemini-2.0-flash", config=config, contents=contents
    )

    # 📦 If the response includes tool calls, handle them
    if response.candidates[0].content.parts[0]:
        tool_call = response.candidates[0].content.parts

        # 🔧 Execute tool call and update session history
        session_history = executeToolCall(tool_call, chat_history=contents)

        # 🤖 Second model call with full session history after tool execution
        response = client.models.generate_content(
            model="gemini-2.0-flash", contents=session_history
        )

        response = response.text
    else:
        # ✉️ If no tool call, just return the model's direct response
        response = response.text

    # 🧾 Record the entire interaction for logging or evaluation
    row = {
        "query": query,
        "output":response,
        "messageHistory": f'{session_history}',
        "tools": tool_descriptions,
    }
    results.append(row)


In [8]:
# preview data structure
results[0]

{'query': 'Show me the menu ',
 'output': "Okay, here's the menu:\n\n*   **Burger:**\n    *   Price: 150\n    *   Stock: 10\n    *   Description: Delicious beef burger\n*   **Pizza:**\n    *   Price: 300\n    *   Stock: 5\n    *   Description: Cheesy pepperoni pizza\n*   **Pasta:**\n    *   Price: 250\n    *   Stock: 8\n    *   Description: Creamy alfredo pasta\n*   **Coke:**\n    *   Price: 50\n    *   Stock: 20\n    *   Description: Refreshing soft drink\n*   **Sandwich:**\n    *   Price: 120\n    *   Stock: 15\n    *   Description: Grilled cheese sandwich\n*   **Fries:**\n    *   Price: 100\n    *   Stock: 12\n    *   Description: Crispy golden french fries\n*   **Mojito:**\n    *   Price: 180\n    *   Stock: 10\n    *   Description: Cool mint mojito\n*   **Coffee:**\n    *   Price: 120\n    *   Stock: 20\n    *   Description: Hot brewed coffee\n*   **Tea:**\n    *   Price: 80\n    *   Stock: 25\n    *   Description: Refreshing herbal tea\n",
 'messageHistory': '[Content(parts=[Part

# **🧠 Evaluate Agent Responses using LlumoClient**
- Uses the `llumo` Python library to **evaluate LLM-generated responses**
- `LlumoClient` is used to score responses on criteria such as:
  - Tool usage correctness
  - Overall quality, completeness and correctness
- Helps analyze and benchmark the performance of the conversational agent



In [14]:
from llumo import LlumoClient

# Initialize the LlumoClient with your API key
client = LlumoClient(api_key=llumo_key)

# Evaluate agent responses based on queries and outputs
resultdf = client.evaluateAgentResponses(
    data = results, # input data
    evals = ['Tool Reliability','Stepwise Progression','Tool Selection Accuracy','Final Task Alignment'], # tool eval metrics
    getDataFrame=True,  # Return result as a DataFrame (True) or dictionary (False)
    createExperiment=False  # When True, creates an experiment (no result object returned here)

)


Processing Batches: 100%|██████████| 4/4 [00:13<00:00,  3.47s/batch]


# **📊 View Evaluation Result Table**


In [15]:
resultdf

,query,output,messageHistory,tools,Tool Reliability,Tool Reliability Reason,Stepwise Progression,Stepwise Progression Reason,Tool Selection Accuracy,Tool Selection Accuracy Reason,Final Task Alignment,Final Task Alignment Reason
0,Show me the menu,"Okay, here's the menu:\n\n* **Burger:**\n ...","[Content(parts=[Part(video_metadata=None, thou...","{'getMenu': 'Get the restaurant menu.', 'addTo...",99,The 'getMenu' tool successfully executed and r...,100,"The tool 'getMenu' was called, which is releva...",99,The user requested the menu. The assistant us...,100,The tool's response directly answers the user'...
1,Add 2 burgers to my cart,OK. I've added 2 burgers to your cart. Your ca...,"[Content(parts=[Part(video_metadata=None, thou...","{'getMenu': 'Get the restaurant menu.', 'addTo...",100,The `addToCart` tool successfully added 2 burg...,99,The tool 'addToCart' is relevant to the user's...,100,The user requested adding burgers to the cart....,99,The user requested adding two burgers to their...
2,Show me the order history,There are no past orders in your order history.\n,"[Content(parts=[Part(video_metadata=None, thou...","{'getMenu': 'Get the restaurant menu.', 'addTo...",100,The tool 'viewOrderHistory' executed successfu...,100,The tool 'viewOrderHistory' is directly releva...,100,The user requested order history. The assista...,100,The tool successfully executed the function 'v...
